<div style="
    background-color:#1f4e78;
    padding:30px;
    border-radius:15px;
    text-align:center;
    color:white;
">

<h1 style="margin:0; font-size:38px;">
    New York City Airbnb Open Data
</h1>

</div>

# NYC Airbnb Data Analysis

## Introduction

This project analyzes Airbnb listings in New York City to explore pricing, room types, neighborhoods, reviews, availability, and host activity.

The analysis will focus on understanding the dataset, cleaning and preparing the data, performing exploratory data analysis (EDA), extracting meaningful business insights, and preparing the final dataset for visualization in Power BI.


# Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

# Load Data

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/dgomonov/new-york-city-airbnb-open-data/AB_NYC_2019.csv")

# Show first 30 rows

In [ ]:
df.head(30)

In [ ]:
df.shape

- 48895 Rows
- 16 Columns

In [ ]:
df.info()

## Dataset Columns Overview

The dataset contains 16 columns describing Airbnb listings, hosts, locations, pricing, reviews, and availability in New York City.

| Column | Description |
|---|---|
| `id` | Unique identifier for each Airbnb listing. |
| `name` | Name or title of the Airbnb listing. |
| `host_id` | Unique identifier for the host. |
| `host_name` | Name of the host. |
| `neighbourhood_group` | NYC borough where the listing is located. |
| `neighbourhood` | Specific neighborhood of the listing. |
| `latitude` | Geographic latitude of the listing. |
| `longitude` | Geographic longitude of the listing. |
| `room_type` | Type of accommodation offered. |
| `price` | Price of the listing per night in USD. |
| `minimum_nights` | Minimum number of nights required for a booking. |
| `number_of_reviews` | Total number of reviews received by the listing. |
| `last_review` | Date of the most recent review. |
| `reviews_per_month` | Average number of reviews received per month. |
| `calculated_host_listings_count` | Number of listings owned by the same host. |
| `availability_365` | Number of days the listing is available during the year. |

# Histogram

In [ ]:
import matplotlib.pyplot as plt

numerical_columns = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365"
]

for col in numerical_columns:
    plt.figure(figsize=(10, 5))
    
    plt.hist(df[col].dropna(), bins=50)
    
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    
    plt.show()

# Heatmap

In [ ]:
corr = df[numerical_columns].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    aspect="auto",
    title="Correlation Heatmap of Numerical Variables"
)

fig.update_layout(
    template="plotly_white",
    height=600
)

fig.show()

<div style="
    background-color:#1f4e78;
    padding:30px;
    border-radius:15px;
    text-align:center;
    color:white;
">

<h1 style="margin:0; font-size:38px;">
    Data Quality Assessment
</h1>

</div>

# Check Nulls

In [ ]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df)) * 100
})

missing = missing.sort_values(
    by="Missing Percentage",
    ascending=False
)

missing

### Missing Values

We first check the number and percentage of missing values in each column.

Understanding missing data is important because different columns may require different treatment depending on the meaning of the missing values.

In [ ]:
missing[missing["Missing Values"] > 0]

### Missing Values Assessment

The dataset contains missing values in four columns: **last_review**, **reviews_per_month**, **host_name**, and **name**.

The largest amount of missing data appears in **last_review** and **reviews_per_month**, with 10,052 missing values in each column (20.56%). These missing values are likely related to listings that have not received any reviews.

The missing values in **host_name** and **name** are very limited, representing less than 0.05% of the dataset.

No missing values will be handled at this stage. Treatment decisions will be made during the Data Cleaning phase based on the meaning and analytical importance of each column.

# Check duplicates

In [ ]:
df.duplicated().sum()

### Duplicate Check Result

The dataset contains **0 duplicate rows**, indicating that there are no fully duplicated records. Therefore, no duplicate records need to be removed at this stage.

# Check unique values

In [ ]:
df.dtypes

### Data Types Assessment

Most columns have appropriate data types for analysis. However, the **last_review** column is currently stored as **object** even though it represents dates. It will be converted to a datetime format during the data cleaning stage.

# Categorical variables

In [ ]:
categorical_columns = [
    "neighbourhood_group",
    "neighbourhood",
    "room_type"
]

for col in categorical_columns:
    print(f"\n{col}")
    print("Unique values:", df[col].nunique())

In [ ]:
df["neighbourhood_group"].value_counts()

In [ ]:
df["room_type"].value_counts()

### Categorical Values Assessment

The categorical variables contain consistent and expected values.

The dataset covers all five New York City boroughs: Manhattan, Brooklyn, Queens, Bronx, and Staten Island. There are 221 unique neighborhoods and three room types: Entire home/apt, Private room, and Shared room.

No unexpected or inconsistent categories were identified, so no categorical values require correction at this stage.

# Numerical variables

In [ ]:
numerical_columns = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365"
]

df[numerical_columns].describe()

### Numerical Variables Assessment

The numerical summary reveals several values that require further investigation before data cleaning.

The **price** column contains a minimum value of 0 and a maximum value of 10,000, which may indicate invalid or extreme values.

The **minimum_nights** column has a maximum value of 1,250 days, which is unusually high and requires further investigation.

The **reviews_per_month** column contains missing values and has a maximum value of 58.5, while **calculated_host_listings_count** reaches a maximum of 327 listings for a single host.

The **availability_365** column ranges from 0 to 365, which is consistent with its definition.

These observations will be investigated further before deciding which values should be treated as invalid values or outliers.

In [ ]:
print("Zero price listings:", (df["price"] == 0).sum())
print("Percentage:", (df["price"] == 0).mean() * 100)

### Zero Price Investigation

The **price** column contains **11** listings with a price of *0*, representing only **0.02%** of the dataset.

Since **price** represents the nightly listing price, zero-priced listings are considered invalid for pricing analysis. Due to their very small proportion, these records can be removed during the data cleaning stage without significantly affecting the dataset.

In [ ]:
print("Listings with minimum_nights > 365:",
      (df["minimum_nights"] > 365).sum())

print("Percentage:",
      (df["minimum_nights"] > 365).mean() * 100)

### Minimum Nights Investigation

The **minimum_nights** column contains **14** listings with a minimum stay greater than **365** days, representing only **0.03%** of the dataset.

These values are extremely high compared with typical Airbnb booking periods and may negatively affect the analysis of minimum stay requirements. They will be investigated and handled during the data cleaning stage.

In [ ]:
df.loc[df["minimum_nights"] > 365, 
       ["id", "name", "minimum_nights", "price", "room_type"]]

### Extreme Minimum Stay Values

The investigation identified 14 listings with **minimum_nights** greater than 365 days. Some listings have extremely high values such as 500, 999, and 1,250 days.

These values are highly unusual for Airbnb stays and can distort statistical analysis related to minimum booking requirements. Therefore, these records will be treated as extreme values and removed during the data cleaning stage.

In [ ]:
df["calculated_host_listings_count"].describe()

In [ ]:
print(
    "Listings with host listing count > 100:",
    (df["calculated_host_listings_count"] > 100).sum()
)

### Host Listing Count Investigation

The **calculated_host_listings_count** column contains 783 listings where the host has more than 100 listings.

Although this value is relatively high, it does not necessarily indicate an error. Professional hosts or property management companies may manage a large number of Airbnb listings.

Therefore, these values will be considered valid unless further analysis identifies inconsistencies.

In [ ]:
invalid_availability = df[
    (df["availability_365"] < 0) |
    (df["availability_365"] > 365)
]

print("Invalid availability values:", len(invalid_availability))

### Availability Validation

The **availability_365** column contains no invalid values. All values fall within the expected range of 0 to 365 days, so no correction is required for this column.

<div style="
    background-color:#1f4e78;
    padding:30px;
    border-radius:15px;
    text-align:center;
    color:white;
">

<h1 style="margin:0; font-size:38px;">
    Data Cleaning
</h1>

</div>

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean

In [ ]:
print("Shape before cleaning:", df_clean.shape)

## 1- Handling Missing Values in reviews_per_month

In [ ]:
df_clean["reviews_per_month"] = df_clean["reviews_per_month"].fillna(0)

### Handling Missing Values in **reviews_per_month**

The missing values in `reviews_per_month` correspond to listings with no review activity.

Since these listings have no reviews, their monthly review count can logically be represented as 0 instead of leaving the values missing.

In [ ]:
df_clean["reviews_per_month"].isna().sum()

## 2- Convert last_review to Datetime

In [ ]:
df_clean["last_review"] = pd.to_datetime( df_clean["last_review"])

In [ ]:
df_clean["last_review"].dtype

###  Date Type Conversion Result

The **last_review** column was successfully converted from **object** to **datetime64[ns]**, allowing date-based analysis and calculations to be performed correctly.

## 3- Handle Missing host_name

In [ ]:
df_clean["host_name"] = df_clean["host_name"].fillna("Unknown")

### Handling Missing `host_name`

Only 21 listings have a missing `host_name`, representing a very small portion of the dataset.

Instead of removing these listings, the missing host names will be replaced with `Unknown` to preserve the records for further analysis.

## 4- Handle missing listing name

In [ ]:
df_clean["name"] = df_clean["name"].fillna("Unknown Listing")

### Handling Missing `name`

Only 16 listings have a missing listing name. Since removing these records would result in unnecessary data loss, the missing values will be replaced with `Unknown Listing`.

## 5- Remove zero prices

In [ ]:
df_clean = df_clean[df_clean["price"] > 0]

In [ ]:
print("Zero price values:", (df_clean["price"] == 0).sum())

### Handling Invalid Prices

The `price` column contains 11 listings with a value of 0. Since the column represents the nightly Airbnb price, a zero price is considered invalid for this analysis.

These records will be removed because they represent a negligible portion of the dataset.

In [ ]:
df_clean.shape

In [ ]:
df_clean.isnull().sum()

In [ ]:
print("Shape:", df_clean.shape)
print("\nMissing Values:")
print(df_clean.isnull().sum())

print("\nDuplicates:")
print(df_clean.duplicated().sum())

print("\nData Types:")
print(df_clean.dtypes)

##  Final Cleaning Validation

The cleaned dataset was validated after applying the data cleaning steps.

The final dataset contains 48,884 records and 16 columns. No duplicate rows remain, missing values were handled appropriately, and the `last_review` column was successfully converted to datetime format.

The remaining missing values in `last_review` represent listings without a recorded review date and are therefore retained as `NaT`.

<div style="
    background-color:#1f4e78;
    padding:30px;
    border-radius:15px;
    text-align:center;
    color:white;
">

<h1 style="margin:0; font-size:38px;">
    Feature Engineering
</h1>

</div>

##  Feature Engineering

Feature engineering is the process of creating new meaningful variables from the existing data.

The goal is to make the dataset more useful for analysis by creating features that can help answer business questions and reveal patterns in Airbnb listings.

# 1- has_reviews 

In [ ]:
df_clean["has_reviews"] = df_clean["number_of_reviews"] > 0

In [ ]:
df_clean["has_reviews"].value_counts()

In [ ]:
df_clean["has_reviews"].value_counts(normalize=True) * 100

### Creating `has_reviews`

A binary feature named `has_reviews` was created based on `number_of_reviews`.

- `True`: The listing has at least one review.
- `False`: The listing has no reviews.

This feature will help analyze the difference between listings with and without review activity.

In [ ]:
print("Total rows:", len(df_clean))
print("Total columns:", len(df_clean.columns))

In [ ]:
# 2. Host Portfolio Segment
df_clean["host_segment"] = pd.cut(
    df_clean["calculated_host_listings_count"],
    bins=[0, 1, 5, 10, 50, float("inf")],
    labels=[
        "1 Listing",
        "2-5 Listings",
        "6-10 Listings",
        "11-50 Listings",
        "50+ Listings"
    ]
)

In [ ]:
# Check the new features
df_clean[
    [
        "has_reviews",
        "host_segment"
    ]
].head()

In [ ]:
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])

df_clean.info()

<div style="
    background-color:#1f4e78;
    padding:30px;
    border-radius:15px;
    text-align:center;
    color:white;
">

<h1 style="margin:0; font-size:38px;">
    EDA
</h1>

</div>

In [ ]:
corr = df_clean[numerical_columns].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    aspect="auto",
    title="Correlation Heatmap of Numerical Variables"
)

fig.update_layout(
    template="plotly_white",
    height=600
)

fig.show()

### How are Airbnb listings distributed across NYC neighbourhood groups?

In [ ]:
# Listings distribution by neighbourhood group

borough_counts = (
    df_clean["neighbourhood_group"]
    .value_counts()
    .reset_index()
)

borough_counts.columns = [
    "neighbourhood_group",
    "count"
]

borough_counts["percentage"] = (
    borough_counts["count"]
    / borough_counts["count"].sum()
    * 100
)

borough_counts

In [ ]:
fig = px.bar(
    borough_counts,
    x="neighbourhood_group",
    y="percentage",
    text="percentage",
    title="Airbnb Listings Distribution by Neighbourhood Group"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Neighbourhood Group",
    yaxis_title="Percentage of Listings (%)"
)

fig.show()

### Insight: Airbnb Listings Distribution

Manhattan and Brooklyn dominate the Airbnb market in New York City, accounting for approximately 85.42% of all listings combined.

Manhattan has the largest share with 44.31% of listings, followed by Brooklyn with 41.11%. In comparison, Queens accounts for 11.59%, while the Bronx and Staten Island represent only a small portion of the total listings.

This indicates that Airbnb listings are highly concentrated in Manhattan and Brooklyn.


# 💰 Average Price by Neighbourhood Group

In [ ]:
price_by_borough = (
    df_clean
    .groupby("neighbourhood_group")["price"]
    .agg(
        average_price="mean",
        median_price="median",
        listings="count"
    )
    .reset_index()
)

price_by_borough = price_by_borough.sort_values(
    "average_price",
    ascending=False
)

price_by_borough

- Mean tells us about the overall average price, while Median gives us a more robust view of the typical listing price.

In [ ]:
fig = px.bar(
    price_by_borough,
    x="neighbourhood_group",
    y="average_price",
    text="average_price",
    title="Average Airbnb Price by Neighbourhood Group"
)

fig.update_traces(
    texttemplate="$%{text:.2f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Neighbourhood Group",
    yaxis_title="Average Price per Night ($)"
)

fig.show()

In [ ]:
fig = px.bar(
    price_by_borough,
    x="neighbourhood_group",
    y="median_price",
    text="median_price",
    title="Median Airbnb Price by Neighbourhood Group"
)

fig.update_traces(
    texttemplate="$%{text:.2f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Neighbourhood Group",
    yaxis_title="Median Price per Night ($)"
)

fig.show()

### Insight: Price Variation Across NYC

Manhattan has the highest average Airbnb listing price at approximately **196.88 per night, followed by Brooklyn at 124.44**. The Bronx has the lowest average price at approximately **$87.58**.

The median price is lower than the average price across all neighbourhood groups, suggesting that high-priced listings influence the average price.

Although Staten Island has a higher average price than Queens, it has a much smaller number of listings (373 vs. 5,666), so this difference should be interpreted with caution.

# 💰 Room Type × Price

In [ ]:
price_by_room = (
    df_clean
    .groupby("room_type")["price"]
    .agg(
        average_price="mean",
        median_price="median",
        listings="count"
    )
    .reset_index()
)

price_by_room = price_by_room.sort_values(
    "average_price",
    ascending=False
)

price_by_room

In [ ]:
fig = px.bar(
    price_by_room,
    x="room_type",
    y="average_price",
    text="average_price",
    title="Average Airbnb Price by Room Type"
)

fig.update_traces(
    texttemplate="$%{text:.2f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Room Type",
    yaxis_title="Average Price per Night ($)"
)

fig.show()

In [ ]:
fig = px.bar(
    price_by_room,
    x="room_type",
    y="median_price",
    text="median_price",
    title="Median Airbnb Price by Room Type"
)

fig.update_traces(
    texttemplate="$%{text:.2f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Room Type",
    yaxis_title="Median Price per Night ($)"
)

fig.show()

In [ ]:
room_distribution = (
    df_clean["room_type"]
    .value_counts()
    .reset_index()
)

room_distribution.columns = [
    "room_type",
    "listings"
]

room_distribution["percentage"] = (
    room_distribution["listings"]
    / len(df_clean)
    * 100
)

room_distribution["percentage"] = (
    room_distribution["percentage"].round(2)
)

room_distribution

In [ ]:
fig = px.pie(
    room_distribution,
    names="room_type",
    values="listings",
    title="Distribution of Airbnb Listings by Room Type"
)

fig.update_traces(
    textinfo="label+percent",
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Listings: %{value}<br>"
        "Percentage: %{percent}<extra></extra>"
    )
)

fig.show()

# Room Type × Neighbourhood Group

In [ ]:
room_by_borough = (
    df_clean
    .groupby(["neighbourhood_group", "room_type"])
    .size()
    .reset_index(name="listings")
)

room_by_borough["percentage"] = (
    room_by_borough["listings"]
    / room_by_borough.groupby("neighbourhood_group")["listings"].transform("sum")
    * 100
)

room_by_borough["percentage"] = room_by_borough["percentage"].round(2)

room_by_borough

### Insight: Room Type Distribution Across Neighbourhood Groups

Room type composition varies considerably across NYC neighbourhood groups.

Manhattan is dominated by Entire home/apt listings, representing 60.93% of its listings, while Private rooms account for 36.85%.

Brooklyn has a more balanced distribution, with Private rooms representing 50.39% and Entire home/apt listings representing 47.56%.

The Bronx and Queens show similar patterns, with Private rooms accounting for approximately 60% of their listings.

Shared rooms are the least common room type across all neighbourhood groups, representing less than 6% in every area.

In [ ]:
fig = px.bar(
    room_by_borough,
    x="neighbourhood_group",
    y="listings",
    color="room_type",
    barmode="group",
    title="Room Type Distribution Across Neighbourhood Groups",
    text="listings"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Neighbourhood Group",
    yaxis_title="Number of Listings"
)

fig.show()

# Price × Neighbourhood Group × Room Type

In [ ]:
price_room_borough = (
    df_clean
    .groupby(["neighbourhood_group", "room_type"])["price"]
    .agg(
        average_price="mean",
        median_price="median",
        listings="count"
    )
    .reset_index()
)

price_room_borough

In [ ]:
fig = px.bar(
    price_room_borough,
    x="neighbourhood_group",
    y="average_price",
    color="room_type",
    barmode="group",
    title="Average Price by Room Type and Neighbourhood Group"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Neighbourhood Group",
    yaxis_title="Average Price per Night ($)"
)

fig.show()

# Review activity

### Which room types receive more reviews?

In [ ]:
reviews_by_room = (
    df_clean
    .groupby("room_type")
    .agg(
        average_reviews=("number_of_reviews", "mean"),
        median_reviews=("number_of_reviews", "median"),
        total_reviews=("number_of_reviews", "sum"),
        listings=("id", "count")
    )
    .reset_index()
)

reviews_by_room

# Availability by Room Type

### Which room types are more available throughout the year?

In [ ]:
availability_by_room = (
    df_clean
    .groupby("room_type")["availability_365"]
    .agg(
        average_availability="mean",
        median_availability="median"
    )
    .reset_index()
)

availability_by_room

# 📊 Reviews vs Availability

### Is there a relationship between listing availability and review activity?

In [ ]:
fig = px.scatter(
    df_clean,
    x="availability_365",
    y="number_of_reviews",
    color="room_type",
    opacity=0.5,
    title="Availability vs Number of Reviews",
    labels={
        "availability_365": "Available Days (365)",
        "number_of_reviews": "Number of Reviews"
    }
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

In [ ]:
df_clean[
    ["availability_365", "number_of_reviews"]
].corr()

### Insight: Availability vs Review Activity

The Pearson correlation between `availability_365` and `number_of_reviews` is approximately 0.172, indicating a weak positive relationship.

This suggests that listings with higher availability tend to have slightly more reviews. However, the relationship is weak, so availability alone is not a strong predictor of review activity.

It is important to note that correlation does not imply causation, and other factors such as price, room type, location, and listing age may influence the number of reviews.

# Top 10 Neighbourhoods 🔝

In [ ]:
top_neighbourhoods = (
    df_clean["neighbourhood"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_neighbourhoods.columns = [
    "neighbourhood",
    "listings"
]

top_neighbourhoods

## Top 10 Neighbourhoods by Number of Listings

This analysis identifies the neighbourhoods with the highest number of Airbnb listings in New York City.

The results show that **Williamsburg** has the largest number of listings with **3,919 listings**, followed by **Bedford-Stuyvesant** with **3,710 listings** and **Harlem** with **2,658 listings**.

These neighbourhoods may represent highly active Airbnb markets and can be further investigated in terms of **average price, room type, and availability**.

In [ ]:
fig = px.bar(
    top_neighbourhoods.sort_values("listings"),
    x="listings",
    y="neighbourhood",
    orientation="h",
    text="listings",
    title="Top 10 Neighbourhoods by Number of Listings"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Number of Listings",
    yaxis_title="Neighbourhood"
)

fig.show()

In [ ]:
neighbourhood_analysis = (
    df_clean.groupby("neighbourhood")
    .agg(
        listings=("id", "count"),
        median_price=("price", "median"),
        avg_reviews=("number_of_reviews", "mean"),
        avg_availability=("availability_365", "mean")
    )
    .reset_index()
)

neighbourhood_analysis = neighbourhood_analysis.sort_values(
    "avg_reviews",
    ascending=False
)

neighbourhood_analysis.head(10)

In [ ]:
top_neighbourhoods = neighbourhood_analysis.head(10)

fig = px.bar(
    top_neighbourhoods,
    x="neighbourhood",
    y="avg_reviews",
    text="avg_reviews",
    title="Top 10 Neighbourhoods by Average Reviews",
    labels={
        "neighbourhood": "Neighbourhood",
        "avg_reviews": "Average Reviews per Listing"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_tickangle=-45
)

fig.show()

## Insight: Neighbourhood Opportunity

Neighbourhood-level analysis reveals differences in listing activity, pricing, review activity, and availability across NYC.

Rather than focusing only on the number of listings, average reviews per listing provides an additional indicator of listing engagement within each neighbourhood.

Neighbourhoods with relatively high review activity may indicate stronger customer engagement, while areas with fewer listings and strong review activity could represent potential opportunities for new hosts.

However, reviews should be treated as an engagement indicator rather than a direct measure of bookings or revenue.

# Hosts Concentration

### Is the Airbnb market dominated by a small number of hosts?

In [ ]:
top_hosts = (
    df_clean
    .groupby(["host_id", "host_name"])
    .size()
    .reset_index(name="listings")
    .sort_values("listings", ascending=False)
    .head(10)
)

top_hosts

## Top Hosts by Number of Listings

This analysis identifies the hosts who manage the largest number of Airbnb listings.

The results show that **Sonder (NYC)** is the host with the highest number of listings, managing **327 listings**, followed by **Blueground** with **232 listings**.

The presence of hosts with a large number of listings suggests that part of the Airbnb market is managed by **professional hosts or property management companies**, rather than individual hosts.

These hosts can be further analyzed to understand their **pricing strategies, room types, neighbourhood distribution, and availability**.

In [ ]:
fig = px.bar(
    top_hosts.sort_values("listings"),
    x="listings",
    y="host_name",
    orientation="h",
    text="listings",
    title="Top 10 Hosts by Number of Listings"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Number of Listings",
    yaxis_title="Host"
)

fig.show()

# Price Distribution 💰

In [ ]:
fig = px.box(
    df_clean,
    y="price",
    title="Airbnb Price Distribution"
)

fig.update_layout(
    template="plotly_white",
    yaxis_title="Price per Night ($)"
)

fig.show()

In [ ]:
fig = px.box(
    df_clean,
    x="neighbourhood_group",
    y="price",
    color="neighbourhood_group",
    title="Price Distribution by Neighbourhood Group"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Neighbourhood Group",
    yaxis_title="Price per Night ($)",
    showlegend=False
)

fig.show()

In [ ]:
host_distribution = (
    df_clean.groupby("calculated_host_listings_count")
    .size()
    .reset_index(name="hosts")
)

host_distribution.head(20)

In [ ]:
host_segments = pd.cut(
    df_clean["calculated_host_listings_count"],
    bins=[0, 1, 5, 10, 50, float("inf")],
    labels=[
        "1 Listing",
        "2-5 Listings",
        "6-10 Listings",
        "11-50 Listings",
        "50+ Listings"
    ]
)

host_segments.value_counts().sort_index()

In [ ]:
# Count listings in each host segment
host_counts = host_segments.value_counts().sort_index().reset_index()
host_counts.columns = ["Host Segment", "Listings"]

# Calculate percentage
host_counts["Percentage"] = (
    host_counts["Listings"] / host_counts["Listings"].sum() * 100
)

fig = px.bar(
    host_counts,
    x="Host Segment",
    y="Listings",
    text="Listings",
    title="Airbnb Listings by Host Portfolio Size",
    labels={
        "Host Segment": "Host Portfolio Size",
        "Listings": "Number of Listings"
    }
)

fig.update_traces(
    texttemplate="%{text:,}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    height=500
)

fig.show()

# Host Size × Price × Neighbourhood Group

In [ ]:
df_clean["host_segment"] = pd.cut(
    df_clean["calculated_host_listings_count"],
    bins=[0, 1, 5, 10, 50, float("inf")],
    labels=[
        "1 Listing",
        "2-5 Listings",
        "6-10 Listings",
        "11-50 Listings",
        "50+ Listings"
    ]
)

host_price_analysis = (
    df_clean.groupby("host_segment", observed=False)
    .agg(
        listings=("price", "size"),
        avg_price=("price", "mean"),
        median_price=("price", "median"),
        avg_reviews=("number_of_reviews", "mean"),
        avg_availability=("availability_365", "mean")
    )
    .reset_index()
)

host_price_analysis

- Host portfolio size is associated with distinct pricing, availability, and review patterns. Large portfolio hosts (50+ listings) are associated with the highest prices and availability but substantially lower review activity, while smaller multi-listing hosts (2–5 listings) show the highest average review counts at lower median prices.

In [ ]:
import plotly.express as px

fig = px.bar(
    host_price_analysis,
    x="host_segment",
    y="avg_availability",
    text="avg_availability",
    title="Average Availability by Host Portfolio Size",
    labels={
        "host_segment": "Host Portfolio Size",
        "avg_availability": "Average Availability (Days)"
    }
)

fig.update_traces(
    texttemplate="%{text:.0f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    yaxis_range=[0, 365]
)

fig.show()

- Large portfolio hosts maintain substantially higher listing availability, but this is accompanied by much lower review activity. This may indicate lower booking activity, although availability alone cannot be used as a direct measure of occupancy

# Minimum Nights × Price & Reviews

In [ ]:
df_clean["minimum_nights_group"] = pd.cut(
    df_clean["minimum_nights"],
    bins=[0, 1, 3, 7, 30, float("inf")],
    labels=[
        "1 Night",
        "2-3 Nights",
        "4-7 Nights",
        "8-30 Nights",
        "30+ Nights"
    ]
)

minimum_nights_analysis = (
    df_clean.groupby("minimum_nights_group", observed=False)
    .agg(
        listings=("id", "count"),
        median_price=("price", "median"),
        avg_price=("price", "mean"),
        avg_reviews=("number_of_reviews", "mean"),
        avg_availability=("availability_365", "mean")
    )
    .reset_index()
)

minimum_nights_analysis

In [ ]:
fig = px.bar(
    minimum_nights_analysis,
    x="minimum_nights_group",
    y="median_price",
    text="median_price",
    title="Median Price by Minimum Stay Requirement",
    labels={
        "minimum_nights_group": "Minimum Stay",
        "median_price": "Median Price ($)"
    }
)

fig.update_traces(
    texttemplate="$%{text:.0f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

## Insight: Minimum Stay and Pricing

Minimum stay requirements vary across Airbnb listings and are associated with different pricing patterns.

Listings with longer minimum-stay requirements may follow a different pricing strategy from short-stay listings.

Comparing minimum nights with median price and review activity helps identify whether hosts targeting longer stays also tend to use different pricing strategies.

Because minimum stay requirements can also be influenced by listing type and location, this relationship should be interpreted alongside room type and neighbourhood.

# Premium Listings Analysis

In [ ]:
premium_threshold = df_clean["price"].quantile(0.90)

premium_listings = df_clean[
    df_clean["price"] >= premium_threshold
].copy()

premium_threshold

In [ ]:
premium_summary = (
    premium_listings
    .groupby("room_type")
    .agg(
        listings=("id", "count"),
        median_price=("price", "median"),
        avg_reviews=("number_of_reviews", "mean"),
        avg_availability=("availability_365", "mean")
    )
    .reset_index()
)

premium_summary

In [ ]:
fig = px.bar(
    premium_summary,
    x="room_type",
    y="listings",
    text="listings",
    title="Premium Listings by Room Type",
    labels={
        "room_type": "Room Type",
        "listings": "Number of Premium Listings"
    }
)

fig.update_traces(
    texttemplate="%{text:,}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

## Insight: Premium Airbnb Listings

Premium listings were defined as listings within the top 10% of the overall price distribution.

Analyzing these listings separately helps identify the characteristics associated with higher-priced Airbnb properties.

The distribution of premium listings across room types can reveal which accommodation types are most commonly positioned at the higher end of the market.

This analysis can help hosts understand the characteristics of the premium segment and identify potential pricing opportunities.

# Room Type Performance 🛏️

In [ ]:
room_performance = (
    df_clean.groupby("room_type")
    .agg(
        listings=("id", "count"),
        median_price=("price", "median"),
        avg_reviews=("number_of_reviews", "mean"),
        avg_availability=("availability_365", "mean")
    )
    .reset_index()
)

room_performance

In [ ]:
fig = px.bar(
    room_performance,
    x="room_type",
    y="avg_reviews",
    text="avg_reviews",
    title="Average Reviews by Room Type",
    labels={
        "room_type": "Room Type",
        "avg_reviews": "Average Reviews"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

## Insight: Room Type Performance

Room types show different patterns in pricing, review activity, and availability.

Average reviews provide an indication of customer engagement, while median price helps compare the pricing position of each room type.

Combining these metrics provides a broader view of how different accommodation types perform within the Airbnb marketplace.

These differences suggest that room type is an important factor when evaluating pricing and market positioning.

# Professional Hosts vs Individual Hosts 👑


In [ ]:
host_performance = (
    df_clean.groupby("host_segment", observed=False)
    .agg(
        listings=("id", "count"),
        median_price=("price", "median"),
        avg_price=("price", "mean"),
        avg_reviews=("number_of_reviews", "mean"),
        avg_availability=("availability_365", "mean")
    )
    .reset_index()
)

host_performance

In [ ]:
fig = px.bar(
    host_performance,
    x="host_segment",
    y="median_price",
    text="median_price",
    title="Median Price by Host Portfolio Size",
    labels={
        "host_segment": "Host Portfolio Size",
        "median_price": "Median Price ($)"
    }
)

fig.update_traces(
    texttemplate="$%{text:.0f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

In [ ]:
fig = px.bar(
    host_performance,
    x="host_segment",
    y="avg_reviews",
    text="avg_reviews",
    title="Average Reviews by Host Portfolio Size",
    labels={
        "host_segment": "Host Portfolio Size",
        "avg_reviews": "Average Reviews"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

## Insight: Host Portfolio Size and Listing Performance

Host portfolio size is associated with notable differences in pricing, review activity, and availability.

Listings managed by hosts with larger portfolios tend to show higher median prices and substantially higher availability.

At the same time, average review activity decreases among the largest host segments, with the 50+ listings segment showing particularly low average reviews.

This pattern suggests that large-scale hosts may follow a different operating and pricing strategy compared with individual and small-scale hosts.

However, availability and reviews should not be interpreted as direct measures of occupancy or revenue because the dataset does not provide actual booking or revenue data.

### ❓ Where is the market concentrated?

- Manhattan and Brooklyn dominate the NYC Airbnb market, representing 85.42% of all listings.

### ❓ Where are prices highest?

- Manhattan has the highest prices, especially Entire home/apt listings, with an average price of $249.26.

### ❓ Where is customer engagement strongest?

- Private rooms have the highest average review activity at 24.10 reviews per listing.

### ❓ How do host strategies differ?

- Large portfolio hosts charge higher prices and have much higher availability, while small multi-listing hosts show stronger average review activity.

### ❓ Where are potential opportunities?

- East Elmhurst and Springfield Gardens are worth further investigation because they combine meaningful supply with relatively high review activity, while very small neighbourhoods should be treated cautiously.

In [ ]:
# Export cleaned dataset for Power BI

df_clean.to_csv(
    "NYC_Airbnb_Cleaned.csv",
    index=False
)

print("Dataset exported successfully!")

<div style="
    background-color:#1f4e78;
    padding:30px;
    border-radius:15px;
    text-align:center;
    color:white;
">

<h1 style="margin:0; font-size:38px;">
    Thanks
</h1>

</div>